In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Link Prediction using Node2Vec (Undirected Graph)
=================================================
This script builds an undirected graph from an XML file (e.g., oncogene_pathways.xml),
learns node embeddings using Node2Vec, trains a Random Forest classifier to predict
missing edges, and outputs high‑confidence novel interactions. Optionally, it validates
predictions against the STRING database.

Dependencies:
    - networkx, numpy, pandas, matplotlib, scikit-learn
    - node2vec (pip install node2vec)
    - requests (for STRING validation)

Usage:
    python 05_link_prediction_node2vec_undirected.py
"""

import os
import random
import time
import xml.etree.ElementTree as ET
from collections import defaultdict

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from node2vec import Node2Vec
import requests

# Set random seeds for reproducibility
random.seed(12345)
np.random.seed(12345)

# ----------------------------------------------------------------------
# 1. Build graph from XML
# ----------------------------------------------------------------------
def build_graph_from_xml(xml_file):
    """Parse XML and return a directed graph."""
    G = nx.DiGraph()
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for pathway in root.findall('pathway'):
        node_list = pathway.find('nodeList')
        if node_list is not None:
            for node in node_list.findall('node'):
                node_name = node.get('name')
                if node_name:
                    G.add_node(node_name)
        edge_list = pathway.find('edgeList')
        if edge_list is not None:
            for edge in edge_list.findall('edge'):
                start = edge.find('startNode').get('node')
                end = edge.find('endNode').get('node')
                if start and end:
                    G.add_edge(start, end)
    return G

# ----------------------------------------------------------------------
# 2. Node2Vec embedding
# ----------------------------------------------------------------------
def learn_node2vec_embeddings(G_undirected, dimensions=64, walk_length=30,
                              num_walks=200, p=1, q=1, workers=4,
                              window=10, min_count=1, batch_words=4):
    """Learn Node2Vec embeddings for an undirected graph."""
    node2vec = Node2Vec(G_undirected, dimensions=dimensions,
                        walk_length=walk_length, num_walks=num_walks,
                        p=p, q=q, workers=workers)
    model = node2vec.fit(window=window, min_count=min_count, batch_words=batch_words)
    embeddings = {node: model.wv[node] for node in G_undirected.nodes()}
    return embeddings

# ----------------------------------------------------------------------
# 3. Negative sampling with degree matching
# ----------------------------------------------------------------------
def degree_matching_negative_sampling(positive_edges, G_undirected):
    """Sample negative edges matching the degree distribution of positive edges."""
    degree = dict(G_undirected.degree())
    degree_vals = list(degree.values())
    degree_bins = np.percentile(degree_vals, [25, 50, 75])

    def degree_bucket(d):
        if d <= degree_bins[0]:
            return 0
        elif d <= degree_bins[1]:
            return 1
        elif d <= degree_bins[2]:
            return 2
        else:
            return 3

    # Bucket distribution of positive edges
    pos_bucket_dist = defaultdict(int)
    for u, v in positive_edges:
        bu = degree_bucket(degree[u])
        bv = degree_bucket(degree[v])
        pos_bucket_dist[(bu, bv)] += 1

    # Nodes per bucket
    bucket_to_nodes = defaultdict(list)
    for n, d in degree.items():
        bucket_to_nodes[degree_bucket(d)].append(n)

    # Existing edges set for exclusion
    existing_edges_set = set([tuple(sorted(e)) for e in G_undirected.edges()])

    negative_edges = []
    for (bu, bv), count in pos_bucket_dist.items():
        candidates_u = bucket_to_nodes[bu]
        candidates_v = bucket_to_nodes[bv]
        sampled = 0
        attempts = 0
        while sampled < count and attempts < count * 20:
            u = random.choice(candidates_u)
            v = random.choice(candidates_v)
            if u != v and (u, v) not in existing_edges_set and (v, u) not in existing_edges_set:
                negative_edges.append((u, v))
                sampled += 1
            attempts += 1
        if sampled < count:
            print(f"Warning: bucket ({bu},{bv}) only sampled {sampled}/{count} negatives")
    return negative_edges

# ----------------------------------------------------------------------
# 4. Edge feature: Hadamard product
# ----------------------------------------------------------------------
def hadamard_product(emb, u, v):
    """Element‑wise product of two node embeddings."""
    return emb[u] * emb[v]

# ----------------------------------------------------------------------
# 5. Train Random Forest classifier
# ----------------------------------------------------------------------
def train_classifier(X_train, y_train, n_estimators=100, random_state=42, n_jobs=-1):
    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state, n_jobs=n_jobs)
    clf.fit(X_train, y_train)
    return clf

def evaluate_classifier(clf, X_test, y_test):
    y_pred_prob = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_prob)
    y_pred = clf.predict(X_test)
    report = classification_report(y_test, y_pred)
    return auc, report

# ----------------------------------------------------------------------
# 6. STRING validation (optional)
# ----------------------------------------------------------------------
def check_string_interaction(protein1, protein2, species=9606, timeout=10):
    """Query STRING database to check if an interaction is known."""
    # Normalize names (uppercase first letter, but keep typical symbols)
    p1_std = protein1.strip()
    p2_std = protein2.strip()
    url = "https://string-db.org/api/json/network"
    identifiers = f"{p1_std}\n{p2_std}"
    params = {
        "identifiers": identifiers,
        "species": species,
        "required_score": 0,
    }
    try:
        resp = requests.post(url, data=params, timeout=timeout)
        if resp.status_code != 200:
            return False, 0.0
        data = resp.json()
        for edge in data:
            a = edge.get('preferredName_A', '')
            b = edge.get('preferredName_B', '')
            if (a.upper() == p1_std.upper() and b.upper() == p2_std.upper()) or \
               (a.upper() == p2_std.upper() and b.upper() == p1_std.upper()):
                score = float(edge.get('score', 0)) / 1000.0
                return True, score
        return False, 0.0
    except Exception:
        return False, 0.0

def validate_predictions(pred_df, n=100, delay=0.5):
    """Validate top n predictions using STRING."""
    results = []
    print(f"Validating top {n} predictions with STRING...")
    for idx, row in pred_df.head(n).iterrows():
        p1, p2 = row['node1'], row['node2']
        known, score = check_string_interaction(p1, p2)
        results.append({'node1': p1, 'node2': p2, 'known': known, 'string_score': score})
        if (idx + 1) % 20 == 0:
            print(f"Processed {idx+1}/{n}")
        time.sleep(delay)
    return pd.DataFrame(results)

# ----------------------------------------------------------------------
# 7. Predict all missing edges
# ----------------------------------------------------------------------
def predict_all_missing_edges(G_undirected, embeddings, clf, batch_size=10000):
    """Predict probabilities for all non‑connected node pairs."""
    nodes = list(G_undirected.nodes())
    existing_edges_set = set([tuple(sorted(e)) for e in G_undirected.edges()])
    all_pairs = []
    for i in range(len(nodes)):
        u = nodes[i]
        for v in nodes[i+1:]:
            if (u, v) not in existing_edges_set:
                all_pairs.append((u, v))
    print(f"Total non‑edge pairs to predict: {len(all_pairs)}")

    pred_results = []
    for start in range(0, len(all_pairs), batch_size):
        batch = all_pairs[start:start+batch_size]
        X_batch = np.array([hadamard_product(embeddings, u, v) for u, v in batch])
        prob = clf.predict_proba(X_batch)[:, 1]
        for (u, v), p in zip(batch, prob):
            pred_results.append((u, v, p))
        if (start + batch_size) % 50000 == 0:
            print(f"Predicted {start+batch_size} pairs")
    pred_df = pd.DataFrame(pred_results, columns=['node1', 'node2', 'probability'])
    pred_df = pred_df.sort_values('probability', ascending=False).reset_index(drop=True)
    return pred_df

# ----------------------------------------------------------------------
# 8. Main pipeline
# ----------------------------------------------------------------------
def main():
    # Input file
    xml_file = "oncogene_pathways.xml"
    if not os.path.exists(xml_file):
        raise FileNotFoundError(f"Input file {xml_file} not found. Please run previous steps first.")

    # Build graph
    G_dir = build_graph_from_xml(xml_file)
    G_und = G_dir.to_undirected()
    print(f"Original directed graph: {G_dir.number_of_nodes()} nodes, {G_dir.number_of_edges()} edges")
    print(f"Undirected graph: {G_und.number_of_nodes()} nodes, {G_und.number_of_edges()} edges")

    # Node2Vec embeddings
    print("Learning Node2Vec embeddings...")
    embeddings = learn_node2vec_embeddings(G_und)
    print(f"Embedding dimension: {len(list(embeddings.values())[0])}")

    # Positive edges
    positive_edges = list(G_und.edges())
    positive_edges = list(set([tuple(sorted(e)) for e in positive_edges]))
    print(f"Positive edges (undirected, unique): {len(positive_edges)}")

    # Negative sampling
    print("Sampling negative edges with degree matching...")
    negative_edges = degree_matching_negative_sampling(positive_edges, G_und)
    print(f"Negative edges sampled: {len(negative_edges)}")

    # Build feature matrix
    X_pos = np.array([hadamard_product(embeddings, u, v) for u, v in positive_edges])
    X_neg = np.array([hadamard_product(embeddings, u, v) for u, v in negative_edges])
    y_pos = np.ones(len(positive_edges))
    y_neg = np.zeros(len(negative_edges))
    X = np.vstack([X_pos, X_neg])
    y = np.hstack([y_pos, y_neg])

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

    # Train classifier
    print("Training Random Forest classifier...")
    clf = train_classifier(X_train, y_train)
    auc, report = evaluate_classifier(clf, X_test, y_test)
    print(f"Test AUC: {auc:.4f}")
    print(report)

    # Predict all missing edges
    print("\nPredicting all missing edges...")
    pred_df = predict_all_missing_edges(G_und, embeddings, clf)

    # Save top predictions
    top_n = 1000
    output_file = "node2vec_link_predictions_top1000.csv"
    pred_df.head(top_n).to_csv(output_file, index=False)
    print(f"Saved top {top_n} predictions to {output_file}")

    # Optional: STRING validation on top 100 predictions
    print("\nValidating top 100 predictions with STRING...")
    val_df = validate_predictions(pred_df, n=100, delay=0.5)
    val_df.to_csv("string_validation_top100.csv", index=False)

    # Compute precision@k
    for k in [10, 20, 50, 100]:
        prec = val_df.head(k)['known'].mean()
        print(f"Precision@{k}: {prec:.3f}")

    # Extract high‑confidence novel candidates
    threshold = 0.99
    candidates = pred_df[(pred_df['probability'] >= threshold) &
                         (~pred_df[['node1', 'node2']].apply(tuple, axis=1).isin(
                             set(zip(val_df['node1'], val_df['node2'])) ))]
    # More precise: use validation results to filter known interactions
    known_pairs = set(zip(val_df[val_df['known']]['node1'], val_df[val_df['known']]['node2']))
    candidates = pred_df[(pred_df['probability'] >= threshold) &
                         (~pred_df[['node1', 'node2']].apply(tuple, axis=1).isin(known_pairs))]
    print(f"\nHigh‑confidence novel candidates (prob ≥ {threshold}): {len(candidates)}")
    candidates.to_csv("novel_candidates.csv", index=False)
    print("Saved novel candidates to novel_candidates.csv")

if __name__ == "__main__":
    main()